# Image Complexity Scorer

This notebook:
1. Installs dependencies
2. Loads a pretrained MobileNetV2 backbone (via `timm`)
3. Defines a NIMA-style complexity scoring head
4. Runs a sanity-check inference
5. Fine-tunes on labeled complexity data (optional)
6. Evaluates the model
7. Prunes the model (optional comparison)
8. Exports to ONNX (FP32, FP16, INT8)
9. Validates ONNX Runtime output matches PyTorch
10. Provides trtexec commands for TensorRT conversion


## 1. Environment Setup
> Verify GPU availability and install required packages.

In [2]:
import subprocess, sys

def run(cmd):
    subprocess.run(cmd, shell=True, check=True)

run('nvidia-smi')
run('pip install -q torch torchvision timm onnx onnxruntime-gpu onnxsim huggingface_hub onnxmltools')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

/bin/sh: 1: nvidia-smi: not found


CalledProcessError: Command 'nvidia-smi' returned non-zero exit status 127.

## 2. Model Definition
> NIMA-style scorer: MobileNetV2 backbone + dropout + linear head + softmax.
> Outputs a probability distribution over scores 1–10.
> Expected score = Σ(score_i × p_i).

In [1]:
import torch
import torch.nn as nn
import timm

class ComplexityScorer(nn.Module):
    def __init__(self, backbone='mobilenetv2_100', num_classes=10, dropout=0.25):
        super().__init__()
        self.backbone = timm.create_model(
            backbone,
            pretrained=True,
            num_classes=0,
            global_pool='avg'
        )
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(feat_dim, num_classes),
            nn.Softmax(dim=1)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

    def predict_score(self, x):
        probs = self.forward(x)
        scores = torch.arange(1, 11, dtype=torch.float32, device=x.device)
        return (probs * scores).sum(dim=1)


model = ComplexityScorer(backbone='mobilenetv2_100', num_classes=10)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total_params:,}')
print(f'Trainable params: {trainable_params:,}')
print(model)

ModuleNotFoundError: No module named 'torch'

## 3. Load Pretrained Weights
> Load a pretrained NIMA checkpoint. The backbone is already ImageNet-pretrained via timm.
> If a full NIMA checkpoint is available, load it here.

In [ ]:
import os

CKPT_URL = 'https://github.com/yunxiaoshi/Neural-IMage-Assessment/releases/download/v1.0/epoch-82.pkl'
CKPT_PATH = 'pretrained_nima.pkl'

if not os.path.exists(CKPT_PATH):
    run(f'wget -q {CKPT_URL} -O {CKPT_PATH}')

raw_state = torch.load(CKPT_PATH, map_location=DEVICE)

# Remap keys to match our model structure
new_state = {}
for k, v in raw_state.items():
    new_key = k.replace('features.', 'backbone.').replace('classifier.', 'head.')
    new_state[new_key] = v

missing, unexpected = model.load_state_dict(new_state, strict=False)
print(f'Missing keys:    {len(missing)}')
print(f'Unexpected keys: {len(unexpected)}')
print('Pretrained weights loaded.')
model.eval()

## 4. Sanity Check Inference
> Run a quick forward pass on a sample image to confirm the model loads and scores correctly.

In [ ]:
from PIL import Image
import torchvision.transforms as T
import requests
from io import BytesIO
import matplotlib.pyplot as plt

TRANSFORM = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

def score_image_url(url):
    resp = requests.get(url, timeout=10)
    img = Image.open(BytesIO(resp.content)).convert('RGB')
    tensor = TRANSFORM(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        score = model.predict_score(tensor)
    return score.item(), img

TEST_URL = 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png'
score, img = score_image_url(TEST_URL)

plt.figure(figsize=(4,4))
plt.imshow(img)
plt.title(f'Complexity Score: {score:.3f} / 10')
plt.axis('off')
plt.show()
print(f'Complexity Score: {score:.4f}')

## 5. Dataset & DataLoader
> Define a dataset class for labeled complexity data (CSV with `filename` and `score` columns).
> Skip this section if you have no labeled data.

In [ ]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader

class ComplexityDataset(Dataset):
    """
    Expects a CSV with columns: filename, score
    score is a float in range [1, 10]
    """
    def __init__(self, csv_path, img_root, transform=None):
        self.df = pd.read_csv(csv_path)
        self.img_root = img_root
        self.transform = transform or TRANSFORM

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_root, row['filename'])).convert('RGB')
        img = self.transform(img)
        score = torch.tensor(float(row['score']), dtype=torch.float32)
        return img, score


HAVE_DATA = False  # Set True if you have a CSV + images

if HAVE_DATA:
    CSV_PATH  = 'data/complexity_labels.csv'
    IMG_ROOT  = 'data/images/'
    dataset   = ComplexityDataset(CSV_PATH, IMG_ROOT)
    train_size = int(0.8 * len(dataset))
    val_size   = len(dataset) - train_size
    train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
    print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')
else:
    print('HAVE_DATA=False — skipping dataset creation.')

## 6. Fine-Tuning
> Two-phase training:
> - Phase 1: freeze backbone, train head only (5 epochs)
> - Phase 2: unfreeze all, fine-tune end-to-end (10 epochs)
> Uses Earth Mover Distance (EMD) loss, standard for NIMA-style training.

In [ ]:
def emd_loss(pred_probs, target_scores, num_classes=10):
    """
    Earth Mover Distance loss.
    pred_probs:    [B, num_classes] softmax output
    target_scores: [B] scalar scores in [1, 10]
    """
    device = pred_probs.device
    scores = torch.arange(1, num_classes + 1, dtype=torch.float32, device=device)
    sigma  = 1.0
    target_dist = torch.exp(
        -0.5 * ((scores.unsqueeze(0) - target_scores.unsqueeze(1)) / sigma) ** 2
    )
    target_dist = target_dist / target_dist.sum(dim=1, keepdim=True)
    pred_cdf   = torch.cumsum(pred_probs,   dim=1)
    target_cdf = torch.cumsum(target_dist,  dim=1)
    return torch.mean((pred_cdf - target_cdf) ** 2)


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for imgs, scores in loader:
        imgs, scores = imgs.to(device), scores.to(device)
        optimizer.zero_grad()
        pred = model(imgs)
        loss = emd_loss(pred, scores)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def validate(model, loader, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for imgs, scores in loader:
            imgs, scores = imgs.to(device), scores.to(device)
            pred = model(imgs)
            total_loss += emd_loss(pred, scores).item()
    return total_loss / len(loader)


if HAVE_DATA:
    os.makedirs('models', exist_ok=True)

    # ── Phase 1: Head only ──────────────────────────────────────────
    for p in model.backbone.parameters():
        p.requires_grad = False

    optimizer = torch.optim.Adam(model.head.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

    print('=== Phase 1: Head-only training ===')
    for epoch in range(5):
        tr_loss  = train_one_epoch(model, train_loader, optimizer, DEVICE)
        val_loss = validate(model, val_loader, DEVICE)
        scheduler.step()
        print(f'Epoch {epoch+1:2d}/5  Train Loss: {tr_loss:.4f}  Val Loss: {val_loss:.4f}')

    # ── Phase 2: Full fine-tune ─────────────────────────────────────
    for p in model.backbone.parameters():
        p.requires_grad = True

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

    print('\n=== Phase 2: Full fine-tuning ===')
    best_val = float('inf')
    for epoch in range(10):
        tr_loss  = train_one_epoch(model, train_loader, optimizer, DEVICE)
        val_loss = validate(model, val_loader, DEVICE)
        scheduler.step()
        print(f'Epoch {epoch+1:2d}/10  Train Loss: {tr_loss:.4f}  Val Loss: {val_loss:.4f}')
        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), 'models/best_complexity.pth')
            print(f'  -> Saved best model (val_loss={best_val:.4f})')

    print(f'\nBest val loss: {best_val:.4f}')
else:
    print('HAVE_DATA=False — skipping fine-tuning.')

## 7. Evaluation
> Report MAE and sample predictions on the validation set.

In [ ]:
import numpy as np

if HAVE_DATA:
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, scores in val_loader:
            imgs = imgs.to(DEVICE)
            pred_scores = model.predict_score(imgs).cpu().numpy()
            all_preds.extend(pred_scores.tolist())
            all_labels.extend(scores.numpy().tolist())

    mae = np.mean(np.abs(np.array(all_preds) - np.array(all_labels)))
    print(f'Validation MAE: {mae:.4f}')

    print('\nSample predictions:')
    for i in range(min(10, len(all_preds))):
        print(f'  True: {all_labels[i]:.2f}  Predicted: {all_preds[i]:.2f}')
else:
    print('HAVE_DATA=False — skipping evaluation.')

## 8. Model Export to ONNX
> We export the model to ONNX to make it easier to deploy and run across different inference
> backends beyond PyTorch. It also enables runtime-specific optimizations such as FP16 or INT8
> acceleration with tools like ONNX Runtime or TensorRT.

In [ ]:
import torch
import onnx
import os

os.makedirs('exports', exist_ok=True)

ONNX_PATH     = 'exports/complexity_scorer.onnx'
ONNX_SIM_PATH = 'exports/complexity_scorer_simplified.onnx'

# Export on CPU for simplicity and compatibility
export_model = model.cpu().eval()

# Example input for export
dummy_input = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    export_model,
    dummy_input,
    ONNX_PATH,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['score_distribution'],
    dynamic_axes={
        'input':              {0: 'batch_size'},
        'score_distribution': {0: 'batch_size'}
    },
    verbose=False
)

print(f'Exported: {ONNX_PATH}')

# Verify
model_onnx = onnx.load(ONNX_PATH)
onnx.checker.check_model(model_onnx)
print('ONNX model check passed.')

# Simplify
run(f'python -m onnxsim {ONNX_PATH} {ONNX_SIM_PATH}')
print(f'Simplified ONNX: {ONNX_SIM_PATH}')

print(f'\nInputs:  {[i.name for i in model_onnx.graph.input]}')
print(f'Outputs: {[o.name for o in model_onnx.graph.output]}')

# Move model back to GPU
model = model.to(DEVICE)

## 9. FP16 ONNX Export
> Convert the FP32 ONNX model to FP16 for GPU-optimized inference.
> Best option for deployment on NVIDIA GPUs.

In [ ]:
from onnxmltools.utils.float16_converter import convert_float_to_float16
from onnxmltools.utils import load_model, save_model

ONNX_FP16_PATH = 'exports/complexity_scorer_fp16.onnx'

fp32_model = load_model(ONNX_SIM_PATH)
fp16_model = convert_float_to_float16(fp32_model, keep_io_types=True)
save_model(fp16_model, ONNX_FP16_PATH)

print(f'FP16 ONNX saved: {ONNX_FP16_PATH}')

## 10. INT8 Static Quantization ONNX Export
> Quantize the model to INT8 using a representative calibration dataset.
> Best for CPU deployment or edge devices.

In [ ]:
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType
import numpy as np

ONNX_INT8_PATH = 'exports/complexity_scorer_int8_static.onnx'

class RandomCalibrationReader(CalibrationDataReader):
    """
    Replace with real images for accurate INT8 calibration.
    Using random data here for demonstration only.
    """
    def __init__(self, n_samples=100):
        self.data = [
            {'input': np.random.randn(1, 3, 224, 224).astype(np.float32)}
            for _ in range(n_samples)
        ]
        self.index = 0

    def get_next(self):
        if self.index >= len(self.data):
            return None
        item = self.data[self.index]
        self.index += 1
        return item


quantize_static(
    model_input=ONNX_SIM_PATH,
    model_output=ONNX_INT8_PATH,
    calibration_data_reader=RandomCalibrationReader(n_samples=100),
    weight_type=QuantType.QInt8
)

print(f'INT8 static ONNX saved: {ONNX_INT8_PATH}')

## 11. Validate ONNX Runtime vs PyTorch
> Confirm that ONNX Runtime output matches PyTorch on the same random input.

In [ ]:
import onnxruntime as ort
import numpy as np

# Check ONNX Runtime installation
print(f'ONNX Runtime version: {ort.__version__}')
providers = ort.get_available_providers()
print(f'Available providers: {providers}')

if 'CUDAExecutionProvider' not in providers:
    print('Warning: CUDAExecutionProvider not available.')
    print('1. Check that CUDA is installed')
    print('2. onnxruntime-gpu is installed')

session = ort.InferenceSession(
    ONNX_SIM_PATH,
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)
print(f'\nUsing provider: {session.get_providers()[0]}')

# Compare outputs
test_input = torch.randn(4, 3, 224, 224)

model.eval()
with torch.no_grad():
    torch_out = model(test_input.to(DEVICE)).cpu().numpy()

ort_out = session.run(
    None,
    {'input': test_input.numpy()}
)[0]

max_diff = np.abs(torch_out - ort_out).max()
print(f'\nMax absolute difference PyTorch vs ONNX Runtime: {max_diff:.6f}')

if max_diff < 1e-4:
    print('✓ Validation passed — outputs match.')
else:
    print('✗ Warning: outputs diverge beyond threshold.')

## 12. Download Exported Models
> Download all three ONNX variants from Colab to your local machine.

In [ ]:
from google.colab import files

print('Downloading ONNX models...')
files.download(ONNX_SIM_PATH)    # FP32 simplified
files.download(ONNX_FP16_PATH)   # FP16
files.download(ONNX_INT8_PATH)   # INT8 static
print('Done.')

## 13. TensorRT Conversion (trtexec)
> Run these commands on your **target machine** after downloading the ONNX files.
> You can run these in ONNX Runtime, TensorRT, OpenVINO, or any runtime that supports
> the ONNX opset you exported with.

In [ ]:
TRTEXEC_GUIDE = '''
# ─────────────────────────────────────────────────────────────────────
# Run on your target machine (not in Colab unless TRT is installed)
# Requires: TensorRT 8.6+ and trtexec in PATH
# ─────────────────────────────────────────────────────────────────────

ONNX="complexity_scorer_simplified.onnx"

# FP32 Engine
trtexec \\
    --onnx=$ONNX \\
    --saveEngine=complexity_fp32.engine \\
    --explicitBatch

# FP16 Engine  (recommended — ~2x speedup, minimal accuracy loss)
trtexec \\
    --onnx=$ONNX \\
    --saveEngine=complexity_fp16.engine \\
    --fp16 \\
    --explicitBatch

# FP16 Engine with dynamic batch sizes
trtexec \\
    --onnx=$ONNX \\
    --saveEngine=complexity_fp16_dynamic.engine \\
    --fp16 \\
    --minShapes=input:1x3x224x224 \\
    --optShapes=input:8x3x224x224 \\
    --maxShapes=input:32x3x224x224 \\
    --explicitBatch

# INT8 Engine  (fastest, requires calibration for best accuracy)
trtexec \\
    --onnx=$ONNX \\
    --saveEngine=complexity_int8.engine \\
    --int8 --fp16 \\
    --explicitBatch

# Benchmark a built engine
trtexec \\
    --loadEngine=complexity_fp16.engine \\
    --batch=8 \\
    --iterations=100 \\
    --warmUp=500 \\
    --duration=10
'''

print(TRTEXEC_GUIDE)

## 14. Recap

In this notebook you:
- Loaded a pretrained MobileNetV2 backbone (ImageNet) with a NIMA-style scoring head
- Optionally fine-tuned on labeled complexity data using EMD loss
- Evaluated with MAE on the validation set
- Exported the model to three ONNX variants:

**Exported models** (in `exports/`):

| File | Description | Best For |
|------|-------------|----------|
| `complexity_scorer_simplified.onnx` | FP32 simplified | Default deployment artifact |
| `complexity_scorer_fp16.onnx` | FP16 converted | GPU inference |
| `complexity_scorer_int8_static.onnx` | INT8 quantized | CPU / edge devices |

You can run these in ONNX Runtime, TensorRT, OpenVINO, or any runtime that supports the ONNX opset you exported with.
